## Import libraries

In [1]:
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv

## Load .env file

In [2]:
load_dotenv()

True

## Database Configuration

In [ ]:
DB_CONFIG = {
    "host":     os.getenv("DB_HOST"),
    "database": os.getenv("DATABASE"),
    "user":     os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD")
}

## Connect to Postgres

In [17]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

## Create tables

In [8]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS banks (
        bank_id   SERIAL PRIMARY KEY,
        bank_name VARCHAR(255) UNIQUE
    );
""")

In [9]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id     SERIAL PRIMARY KEY,
        bank_id INT REFERENCES banks(bank_id),
        review_text       TEXT,
        rating            INT,
        review_date       DATE,
        sentiment_label   VARCHAR(50),
        sentiment_score   NUMERIC(8, 6), 
        identified_theme  VARCHAR(255),
        source            VARCHAR(50)
    );
""")

In [ ]:
conn.commit()

print("Tables created successfully.")

Tables created successfully.


### Load CSV Data

In [18]:
df = pd.read_csv('../data/processed/final_mobile_reviews.csv')
df.head()

,review_id,review_text,rating,date,bank,source,sentiment_label,sentiment_score,identified_theme
0,1,good,5,2026-05-12,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.999816,General Satisfaction
1,2,cbe,1,2026-05-12,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.996601,Other
2,3,good use,5,2026-05-12,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.999846,General Satisfaction
3,4,cbe,4,2026-05-11,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.996601,Other
4,5,best secured,5,2026-05-11,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.999819,General Satisfaction


## INSERT BANKS TABLE

In [ ]:
unique_banks = df['bank'].unique()

for bank in unique_banks:
    cur.execute("""
        INSERT INTO banks (bank_name)
        VALUES (%s)
        ON CONFLICT (bank_name) DO NOTHING;
    """, (bank,))

conn.commit()

## FETCH BANK ID MAPPING

In [20]:
cur.execute("SELECT bank_id, bank_name FROM banks;")
rows = cur.fetchall()

bank_map = {name: bank_id for bank_id, name in rows}

print(bank_map)

{'Commercial Bank of Ethiopia': 1, 'Bank of Abyssinia': 2, 'Dashen Bank': 3}


## Prepare dataframe for insertion

In [ ]:
# MAP BANK NAME → BANK_ID
df['bank_id'] = df['bank'].map(bank_map)
df_sql = df.rename(columns={
    'date': 'review_date'
})

In [ ]:
df_sql

,review_id,review_text,rating,review_date,bank,source,sentiment_label,sentiment_score,identified_theme,bank_id
0,1,good,5,2026-05-12,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.999816,General Satisfaction,1
1,2,cbe,1,2026-05-12,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.996601,Other,1
2,3,good use,5,2026-05-12,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.999846,General Satisfaction,1
3,4,cbe,4,2026-05-11,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.996601,Other,1
4,5,best secured,5,2026-05-11,Commercial Bank of Ethiopia,Google Play,POSITIVE,0.999819,General Satisfaction,1
...,...,...,...,...,...,...,...,...,...,...
1387,1388,user friendly using lite app fast easy remove ...,5,2022-07-18,Dashen Bank,Google Play,NEGATIVE,-0.999198,Transaction Performance,3
1388,1389,nice attractive application developed wonderful,5,2022-07-18,Dashen Bank,Google Play,POSITIVE,0.999883,General Satisfaction,3
1389,1390,amole lite simple kind payment thank dashen bank,5,2022-07-18,Dashen Bank,Google Play,POSITIVE,0.998042,UI & User Experience,3
1390,1391,app,4,2022-07-18,Dashen Bank,Google Play,NEGATIVE,-0.840030,Other,3


## Insert reviews into PostgreSQL

In [ ]:
# INSERT REVIEWS TABLE

for _, row in df_sql.iterrows():

    cur.execute("""
        INSERT INTO reviews (
            review_id,
            bank_id,
            review_text,
            rating,
            review_date,
            sentiment_label,
            sentiment_score,
            identified_theme,
            source
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['review_id'],
        row['bank_id'],
        row['review_text'],
        row['rating'],
        row['review_date'],
        row['sentiment_label'],
        row['sentiment_score'],
        row['identified_theme'],
        row['source']
    ))

conn.commit()

## Close connection

In [24]:
cur.close()
conn.close()

print("Done — connection closed.")

Done — connection closed.
